In [60]:
%pip install datasets evaluate transformers accelerate torch
%pip install sacrebleu

Load the dataset:

In [61]:
from datasets import load_dataset

raw_datasets = load_dataset("xmj2002/Chinese_modern_classical")
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 972467
    })
})


Split the dataset into train/valid/test:

In [62]:
# Split the dataset since it only has 'train'
# We select 12000 samples: 10000 train, 1000 validation, 1000 test
shuffled_dataset = raw_datasets["train"].shuffle(seed=777).select(range(12000))

# Split into train (10000) and rest (2000)
train_testvalid = shuffled_dataset.train_test_split(test_size=2000, seed=777)

# Split rest (2000) into validation (1000) and test (1000)
test_valid = train_testvalid["test"].train_test_split(test_size=1000, seed=777)

from datasets import DatasetDict
raw_datasets = DatasetDict({
    'train': train_testvalid['train'],
    'valid': test_valid['train'],
    'test': test_valid['test']
})
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 10000
    })
    valid: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 1000
    })
})


Load the Wenyanwen model and tokenizer:

In [63]:
from transformers import EncoderDecoderModel, AutoTokenizer
import torch

PRETRAINED = "raynardj/wenyanwen-chinese-translate-to-ancient"
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
model = EncoderDecoderModel.from_pretrained(PRETRAINED)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

Using device: cuda


## Evaluation Setup

In [64]:
from evaluate import load

metric = load("sacrebleu")

In [65]:
import numpy as np

def postprocess_text(preds, labels):
    # Just strip whitespace, don't remove spaces (already removed in inference)
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(predictions, references):
    decoded_preds, decoded_labels = postprocess_text(predictions, references)
    # Use 'zh' tokenizer for Chinese text
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, tokenize="zh")
    result = {"bleu": result["score"]}
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Inference Function

The model requires specific parameters:
- `eos_token_id` must be set to `tokenizer.sep_token_id` for complete translations
- `bos_token_id` set to 101
- `num_beams=3` for better quality

In [66]:
def inference_batch(texts, max_length=128):
    """Translate a batch of modern Chinese texts to classical Chinese"""
    tk_kwargs = dict(
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors='pt'
    )
    
    inputs = tokenizer(texts, **tk_kwargs)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            num_beams=3,
            bos_token_id=101,
            eos_token_id=tokenizer.sep_token_id,
            pad_token_id=tokenizer.pad_token_id,
            max_length=max_length,
        )
    
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # Remove spaces between characters
    return [text.replace(" ", "") for text in decoded]

## Test with a few examples

In [67]:
# Test with a few examples from the dataset
test_samples = raw_datasets["test"].select(range(3))

for i, sample in enumerate(test_samples):
    modern = sample["modern"]
    classical_ref = sample["classical"]
    
    prediction = inference_batch([modern])[0]
    
    print(f"Example {i+1}:")
    print(f"Modern:     {modern}")
    print(f"Reference:  {classical_ref}")
    print(f"Prediction: {prediction}")
    print("-" * 80)

Example 1:
Modern:     此时已过了中午，我说： 为什么不游完此地然后再吃中饭。 
Reference:  时已过午，余曰： 何不了此而后中食。 
Prediction: 时已过午，余曰：何不游此而后中饭。
--------------------------------------------------------------------------------
Example 2:
Modern:     如果说现在所做的改革创新有违背错乱而出现偏差的话，也没有听到有什么明显的证据能够使我的方法不能成立《元嘉历》中将闰余减二就直接用以沿袭旧有的粗疏数据，所以推算的变化情况与天象不符合。
Reference:  若谓今所革创违舛失衷者，未闻显据有以矫夺臣法也。《元嘉历》术，减闰余二，直以袭旧分粗，故进退未合。
Prediction: 若谓今所为创新有悖谬，亦未闻有明证能使臣术不成，《元嘉历》减闰余，直以因旧疏，故推移不合于天。
--------------------------------------------------------------------------------
Example 2:
Modern:     如果说现在所做的改革创新有违背错乱而出现偏差的话，也没有听到有什么明显的证据能够使我的方法不能成立《元嘉历》中将闰余减二就直接用以沿袭旧有的粗疏数据，所以推算的变化情况与天象不符合。
Reference:  若谓今所革创违舛失衷者，未闻显据有以矫夺臣法也。《元嘉历》术，减闰余二，直以袭旧分粗，故进退未合。
Prediction: 若谓今所为创新有悖谬，亦未闻有明证能使臣术不成，《元嘉历》减闰余，直以因旧疏，故推移不合于天。
--------------------------------------------------------------------------------
Example 3:
Modern:     建武二年，纳为皇太子妃，却不被宠爱。
Reference:  建武二年，纳爲皇太子妃而无宠。
Prediction: 建武二年，纳为皇太子妃，而无宠。
----------------------------------------------------------------------

## Full Test Set Evaluation

In [68]:
from tqdm.auto import tqdm

batch_size = 16
test_dataset = raw_datasets["test"]

all_predictions = []
all_references = []

print(f"Translating {len(test_dataset)} test samples...")

for i in tqdm(range(0, len(test_dataset), batch_size), desc="Translating"):
    batch = test_dataset[i:i+batch_size]
    
    # Get modern Chinese texts
    modern_texts = batch["modern"]
    
    # Get predictions
    predictions = inference_batch(modern_texts)
    
    # Store results
    all_predictions.extend(predictions)
    all_references.extend(batch["classical"])

print(f"\nTotal predictions: {len(all_predictions)}")

Translating 1000 test samples...


Translating:   0%|          | 0/63 [00:00<?, ?it/s]


Total predictions: 1000


## Calculate BLEU Score

In [72]:
result = compute_metrics(all_predictions, all_references)
print(f"BLEU score: {result['bleu']}")

BLEU score: 47.4939


In [70]:
# Debug: Check how predictions and references look
print("Sample prediction:", repr(all_predictions[0]))
print("Sample reference:", repr(all_references[0]))
print("\nAfter postprocess:")
preds, labels = postprocess_text(all_predictions[:3], all_references[:3])
for i in range(3):
    print(f"\nPred {i}: {repr(preds[i])}")
    print(f"Ref {i}: {repr(labels[i][0])}")

Sample prediction: '时已过午，余曰：何不游此而后中饭。'
Sample reference: '时已过午，余曰： 何不了此而后中食。 '

After postprocess:

Pred 0: '时已过午，余曰：何不游此而后中饭。'
Ref 0: '时已过午，余曰： 何不了此而后中食。'

Pred 1: '若谓今所为创新有悖谬，亦未闻有明证能使臣术不成，《元嘉历》减闰余，直以因旧疏，故推移不合于天。'
Ref 1: '若谓今所革创违舛失衷者，未闻显据有以矫夺臣法也。《元嘉历》术，减闰余二，直以袭旧分粗，故进退未合。'

Pred 2: '建武二年，纳为皇太子妃，而无宠。'
Ref 2: '建武二年，纳爲皇太子妃而无宠。'


## Display Random Examples

In [71]:
import random

num_examples = 10
print(f"--- Displaying {num_examples} random examples ---\n")

indices = random.sample(range(len(all_predictions)), num_examples)

for idx in indices:
    print(f"Example {idx}:")
    print(f"Source (Modern):       {test_dataset[idx]['modern']}")
    print(f"Reference (Classical): {all_references[idx]}")
    print(f"Prediction:            {all_predictions[idx]}")
    print("-" * 80)

--- Displaying 10 random examples ---

Example 430:
Source (Modern):       十二日，设置大都等路打捕民匠等户的总管府。
Reference (Classical): 癸亥，置大都等路打捕民匠等户总管府。
Prediction:            丙辰，置大都等路打捕民匠等户总管府。
--------------------------------------------------------------------------------
Example 53:
Source (Modern):       她的父亲是河内温人，很早就死了，母亲改嫁为魏郡郑翁的妻子，生下儿子郑惮。
Reference (Classical): 父河内温人，蚤卒，母更嫁为魏郡郑翁妻，生男惲。
Prediction:            其父河内温人，早卒，母更嫁为魏郡郑翁妻，生子。
--------------------------------------------------------------------------------
Example 42:
Source (Modern):       高若呐被罢免，任命狄青为枢密使。
Reference (Classical): 高若讷罢，以狄青为枢密使。
Prediction:            若讷罢，以青为枢密使。
--------------------------------------------------------------------------------
Example 533:
Source (Modern):       但山里出现山精枭阳，水中出现水精罔象，树木中产生木精毕方，井里生出土精坟羊，人们就会感到奇怪了，这是因为平时看得少、听得少，对这方面事物的认识比较浅薄。天下的怪异之事物，只有圣人能认识；利害之间的转化，只有聪明的智者能看透。
Reference (Classical): 老槐生火，久血为燐，人弗怪也。山出枭阳，水生罔象，木生毕方，井生坟羊，人怪之，闻见鲜而识物浅也，天下之怪物，圣人之所独见；利害之反复，知者之所独明达也。
Prediction:            然山出山精枭阳，水出水精罔象